# Sample size và thời lượng từ baseline thật: "chạy 4 tuần là biết"?

Trưởng nhóm CRM của cửa hàng quà tặng online (bộ Online Retail II) nhắn:

> "Mình muốn A/B test **email chào mừng kèm voucher 10%** gửi cho khách mới ngay sau đơn đầu tiên. Mục tiêu
> là kéo khách quay lại mua lần hai. Chạy 4 tuần là biết có hiệu quả không, đúng không?"

Trước khi trả lời "được" hay "không", DA cần ba con số từ dữ liệu thật:

1. **Baseline** của metric chính: tỉ lệ khách mới mua lần hai trong 60 ngày sau đơn đầu.
2. **Lưu lượng**: mỗi tuần có bao nhiêu khách mới đủ điều kiện vào thí nghiệm.
3. **Sample size / MDE**: muốn thấy mức cải thiện X điểm phần trăm thì cần bao nhiêu khách mỗi nhóm, tức bao
   nhiêu tuần; ngược lại, chạy N tuần thì phát hiện được cải thiện nhỏ nhất là bao nhiêu.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/11-stats-experimentation/exercises/basic/02-sample-size-from-baseline -q`

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import norm

## Phần A. Baseline và lưu lượng

`customer_invoices(raw)` đã viết sẵn: từ bảng dòng hàng (cột `invoice`, `quantity`, `price`, `customer_id`,
`invoice_date`), giữ dòng hợp lệ có mã khách (định nghĩa chung của track: bỏ hoá đơn `C...`, bỏ
`quantity <= 0` hoặc `price <= 0`) và gộp về **một dòng mỗi hoá đơn**: `customer_id`, `invoice`,
`invoice_date` (giờ sớm nhất của hoá đơn), `revenue`.

`repeat_rate(invoices, window_days, cohort_start, data_end) -> (rate, n_eligible)`:

- Ngày mua đầu của khách = `invoice_date` nhỏ nhất của khách trong `invoices`.
- Khách **đủ điều kiện** khi ngày mua đầu `>= cohort_start` **và** ngày mua đầu + `window_days` ngày
  `<= data_end` (đã có đủ cửa sổ quan sát; khách mua đầu tuần trước chưa thể "quay lại trong 60 ngày").
- Khách **quay lại** khi có ít nhất một hoá đơn với `invoice_date` **lớn hơn** ngày mua đầu và
  `<=` ngày mua đầu + `window_days` ngày.
- Trả `(tỉ lệ quay lại trong số khách đủ điều kiện, số khách đủ điều kiện)` kiểu `(float, int)`.

`weekly_new_customers(invoices, start, end) -> Series`: số khách có ngày mua đầu trong `[start, end)` theo
**tuần thứ Hai-Chủ nhật** (`to_period("W-SUN")`), index là `Period` tuần tăng dần, tên `new_customers`.
Tuần không có khách mới nào thì không cần xuất hiện.

In [ ]:
def customer_invoices(raw: pd.DataFrame) -> pd.DataFrame:
    """Một dòng mỗi hoá đơn hợp lệ có mã khách: customer_id, invoice, invoice_date, revenue. (Đã viết sẵn.)"""
    valid = raw[~raw["invoice"].astype(str).str.startswith("C") & (raw["quantity"] > 0) & (raw["price"] > 0)
                & raw["customer_id"].notna()]
    valid = valid.assign(invoice=valid["invoice"].astype(str), revenue=valid["quantity"] * valid["price"])
    return valid.groupby(["customer_id", "invoice"], as_index=False).agg(
        invoice_date=("invoice_date", "min"), revenue=("revenue", "sum"))


def repeat_rate(invoices: pd.DataFrame, window_days: int, cohort_start, data_end) -> tuple[float, int]:
    """Tỉ lệ khách mới mua lại trong window_days ngày, chỉ tính khách đã đủ cửa sổ quan sát."""
    first = invoices.groupby("customer_id")["invoice_date"].min().rename("first")
    window = pd.Timedelta(days=window_days)
    eligible = first[(first >= pd.Timestamp(cohort_start)) & (first + window <= pd.Timestamp(data_end))]
    joined = invoices.join(first, on="customer_id")
    later = joined[(joined["invoice_date"] > joined["first"]) & (joined["invoice_date"] <= joined["first"] + window)]
    returned = eligible.index.isin(later["customer_id"].unique())
    return float(returned.mean()), int(len(eligible))


def weekly_new_customers(invoices: pd.DataFrame, start, end) -> pd.Series:
    """Số khách mới mỗi tuần (W-SUN) với ngày mua đầu trong [start, end), tên "new_customers"."""
    first = invoices.groupby("customer_id")["invoice_date"].min()
    first = first[(first >= pd.Timestamp(start)) & (first < pd.Timestamp(end))]
    return first.dt.to_period("W-SUN").value_counts().sort_index().rename("new_customers")

In [ ]:
# explore: nạp Online Retail II (lần đầu ~8 giây, có cache parquet trong thư mục tạm)
import sys
import os
import tempfile
import uuid
from pathlib import Path

import duckdb
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure  # noqa: E402

SHEETS = ("Year 2009-2010", "Year 2010-2011")  # sheet đầu chứa lại 01-09/12/2010 của sheet sau: bỏ phần trùng
CACHE_DIR = Path(tempfile.gettempdir()) / "da-cache"  # cache parquet dùng chung với module 03, không ghi vào data/raw


def _sql_path(path: Path) -> str:
    return str(path).replace("'", "''")


def _sheet_sql(xlsx: Path, sheet: str) -> str:
    return f"""
        SELECT Invoice AS invoice, StockCode AS stock_code, Description AS description,
               CAST(CAST(Quantity AS DOUBLE) AS INTEGER) AS quantity,
               TIMESTAMP '1899-12-30' + to_seconds(CAST(round(CAST(InvoiceDate AS DOUBLE) * 86400) AS BIGINT))
                   AS invoice_date,
               CAST(Price AS DOUBLE) AS price,
               CAST(CAST("Customer ID" AS DOUBLE) AS INTEGER) AS customer_id,
               Country AS country
        FROM read_xlsx('{_sql_path(xlsx)}', sheet = '{sheet}', all_varchar = true)"""


def _excel_extension_missing(exc: duckdb.Error) -> bool:
    """Chỉ lỗi cài/nạp extension excel (máy offline...). Lỗi SQL hay lỗi dữ liệu thì phải nổi lên."""
    kinds = (duckdb.CatalogException, duckdb.IOException, duckdb.HTTPException)
    return isinstance(exc, kinds) and "extension" in str(exc).lower()


def load_online_retail(xlsx: Path) -> pd.DataFrame:
    """Hai sheet Online Retail II gộp (chưa lọc), cột snake_case: invoice, stock_code, description, quantity,
    invoice_date, price, customer_id, country.

    Lần đầu đọc xlsx bằng extension excel của DuckDB (~8 giây; máy không cài được extension thì pandas ~60-90 giây)
    rồi ghi cache parquet vào thư mục tạm của hệ điều hành, khoá theo mtime và kích thước tệp; lần sau ~1 giây.
    """
    stat = xlsx.stat()
    cache = CACHE_DIR / f"online_retail_ii_{stat.st_mtime_ns}_{stat.st_size}.parquet"
    con = duckdb.connect()
    if cache.exists():
        try:
            return con.execute(f"SELECT * FROM read_parquet('{_sql_path(cache)}')").df()
        except duckdb.IOException:  # cache hỏng: đọc lại từ xlsx và ghi đè
            pass
    try:
        con.execute(
            f"CREATE TABLE online_retail AS SELECT * FROM ({_sheet_sql(xlsx, SHEETS[0])}) "
            f"WHERE invoice_date < TIMESTAMP '2010-12-01' UNION ALL {_sheet_sql(xlsx, SHEETS[1])}"
        )
    except duckdb.Error as exc:
        if not _excel_extension_missing(exc):
            raise
        old, new = (pd.read_excel(xlsx, sheet_name=s) for s in SHEETS)
        raw = pd.concat([old[old.InvoiceDate < "2010-12-01"], new], ignore_index=True)
        frame = pd.DataFrame({
            "invoice": raw["Invoice"].astype(str), "stock_code": raw["StockCode"].astype(str),
            "description": raw["Description"], "quantity": raw["Quantity"].astype("int32"),
            "invoice_date": raw["InvoiceDate"], "price": raw["Price"].astype(float),
            "customer_id": raw["Customer ID"].astype("Int32"), "country": raw["Country"],
        })
        con.register("frame", frame)
        con.execute("CREATE TABLE online_retail AS SELECT * FROM frame")
    tmp = CACHE_DIR / f".{cache.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp"
    try:  # ghi atomic: tệp tạm tên riêng rồi rename, an toàn khi nhiều tiến trình cùng ghi
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        con.execute(f"COPY online_retail TO '{_sql_path(tmp)}' (FORMAT parquet)")
        os.replace(tmp, cache)
    except (OSError, duckdb.Error):  # không ghi được cache thì thôi, lần sau đọc lại xlsx
        tmp.unlink(missing_ok=True)
    return con.execute("SELECT * FROM online_retail").df()


raw = load_online_retail(ensure("online-retail-ii") / "online_retail_II.xlsx")
invoices = customer_invoices(raw)
DATA_END = raw["invoice_date"].max()
print(f"{len(raw):,} dòng, {len(invoices):,} hoá đơn có mã khách, {invoices.customer_id.nunique():,} khách; "
      f"dữ liệu tới {DATA_END}")

In [ ]:
# explore: baseline theo cửa sổ, và lưu lượng khách mới mỗi tuần năm 2011
# Cohort từ 2010-01-01: khách "mới" tháng 12/2009 có thể đã mua trước khi dữ liệu bắt đầu (bị cắt trái)
for window in (30, 60, 90):
    rate, n = repeat_rate(invoices, window, "2010-01-01", DATA_END)
    print(f"quay lại trong {window} ngày: {rate:.1%} (n = {n:,} khách đủ cửa sổ)")

weekly = weekly_new_customers(invoices, "2011-01-03", "2011-11-28")  # 47 tuần trọn vẹn
print(f"{len(weekly)} tuần, {weekly.sum():,} khách mới; TB {weekly.mean():.1f}/tuần, trung vị {weekly.median():.0f}, "
      f"thấp nhất {weekly.min()}, cao nhất {weekly.max()}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(weekly.index.start_time, weekly.values, width=5, color="#4c78a8")
ax.axhline(weekly.mean(), color="#e45756", lw=1)
ax.annotate(f"TB {weekly.mean():.1f}/tuần", (weekly.index.start_time[1], weekly.mean() + 2), color="#e45756")
ax.set_ylabel("khách mới / tuần")
ax.set_xlabel("tuần (bắt đầu thứ Hai)")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Mỗi tuần chỉ khoảng 30 khách mới vào được thí nghiệm", loc="left")
plt.show()

## Phần B. Sample size, thời lượng, MDE

Metric là tỉ lệ (khách quay lại / khách được gửi), kiểm định hai phía so hai tỉ lệ. Cỡ mẫu **mỗi nhóm** để
phát hiện chênh lệch tuyệt đối `mde_abs` từ baseline `p0` (tức `p1 = p0 + mde_abs`), với mức ý nghĩa
`alpha` và power `power`:

$$n = \frac{\left(z_{1-\alpha/2}\sqrt{2\bar p(1-\bar p)} + z_{1-\beta}\sqrt{p_0(1-p_0) + p_1(1-p_1)}\right)^2}{\text{MDE}^2}, \qquad \bar p = \frac{p_0 + p_1}{2}$$

với $$z_q$$ là phân vị chuẩn (`norm.ppf(q)`) và $$1-\beta$$ là power. Làm tròn **lên** số nguyên.

- `sample_size_per_arm(p0, mde_abs, alpha=0.05, power=0.8) -> int`: công thức trên.
- `weeks_needed(n_per_arm, weekly_eligible, arms=2, share=1.0, min_weeks=2) -> int`: số **tuần trọn** để gom
  đủ `n_per_arm * arms` người, khi mỗi tuần có `weekly_eligible` người đủ điều kiện và chỉ `share` trong số
  đó được đưa vào thí nghiệm. Làm tròn lên; không ít hơn `min_weeks` (chạy ít nhất hai chu kỳ tuần).
- `mde_for_weeks(p0, weekly_eligible, weeks, arms=2, share=1.0, alpha=0.05, power=0.8) -> float`: MDE tuyệt
  đối **nhỏ nhất** phát hiện được khi chạy `weeks` tuần: số người mỗi nhóm là
  `floor(weeks * weekly_eligible * share / arms)`; tìm `mde_abs` nhỏ nhất trong `(0, 1 - p0)` mà
  `sample_size_per_arm(p0, mde_abs, ...)` không vượt số đó (tìm nhị phân, sai số dưới `1e-5`). Không đạt
  được kể cả khi `mde_abs` gần `1 - p0` thì trả `float("nan")`.

In [ ]:
def sample_size_per_arm(p0: float, mde_abs: float, alpha: float = 0.05, power: float = 0.8) -> int:
    """Số người mỗi nhóm để phát hiện p0 -> p0 + mde_abs (hai phía), làm tròn lên."""
    p1 = p0 + mde_abs
    p_bar = (p0 + p1) / 2
    z_alpha, z_beta = norm.ppf(1 - alpha / 2), norm.ppf(power)
    top = z_alpha * math.sqrt(2 * p_bar * (1 - p_bar)) + z_beta * math.sqrt(p0 * (1 - p0) + p1 * (1 - p1))
    return math.ceil(top**2 / mde_abs**2)


def weeks_needed(n_per_arm: int, weekly_eligible: float, arms: int = 2, share: float = 1.0,
                 min_weeks: int = 2) -> int:
    """Số tuần trọn để đủ n_per_arm * arms người, không ít hơn min_weeks."""
    return max(min_weeks, math.ceil(n_per_arm * arms / (weekly_eligible * share)))


def mde_for_weeks(p0: float, weekly_eligible: float, weeks: int, arms: int = 2, share: float = 1.0,
                  alpha: float = 0.05, power: float = 0.8) -> float:
    """MDE tuyệt đối nhỏ nhất phát hiện được sau `weeks` tuần (tìm nhị phân)."""
    available = math.floor(weeks * weekly_eligible * share / arms)
    lo, hi = 1e-9, 1 - p0 - 1e-9
    if available < 1 or sample_size_per_arm(p0, hi, alpha, power) > available:
        return float("nan")
    while hi - lo > 1e-7:
        mid = (lo + hi) / 2
        if sample_size_per_arm(p0, mid, alpha, power) <= available:
            hi = mid
        else:
            lo = mid
    return hi

In [ ]:
# explore: trả lời "4 tuần là biết?" bằng hai bảng
p0, n_eligible = repeat_rate(invoices, 60, "2010-01-01", DATA_END)
traffic = weekly.mean()
plan = pd.DataFrame([
    {"mde_pp": m * 100, "relative": m / p0, "n_per_arm": sample_size_per_arm(p0, m),
     "weeks": weeks_needed(sample_size_per_arm(p0, m), traffic)}
    for m in (0.02, 0.03, 0.05, 0.10)
])
print(f"baseline 60 ngày p0 = {p0:.4f} (n = {n_eligible:,}); {traffic:.1f} khách mới/tuần")
print(plan.round(3).to_string(index=False))

horizon = pd.Series({w: mde_for_weeks(p0, traffic, w) for w in (4, 8, 12, 26, 52)}, name="mde_abs")
print((horizon * 100).round(1).rename("MDE (điểm %)").to_string())

weeks_grid = range(2, 105)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(list(weeks_grid), [mde_for_weeks(p0, traffic, w) * 100 for w in weeks_grid], color="#4c78a8")
ax.axhline(5, color="#9aa5b1", ls="--", lw=1)
ax.annotate("mục tiêu CRM: +5 điểm %", (60, 5.5), color="#5f6b7a")
ax.set_xlabel("số tuần chạy thí nghiệm")
ax.set_ylabel("MDE tuyệt đối (điểm %)")
ax.set_ylim(0, 30)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Chạy 4 tuần chỉ phát hiện được cải thiện từ ~25 điểm % trở lên", loc="left")
plt.show()

## Đọc kết quả, rồi trả lời trưởng nhóm CRM

1. Baseline 30, 60, 90 ngày khác nhau thế nào? Vì sao phải bỏ khách chưa đủ cửa sổ quan sát (khách mua đầu
   tháng 11/2011)? Vì sao cohort bắt đầu từ 2010-01-01 chứ không phải 2009-12-01?
2. Với khoảng 30 khách mới mỗi tuần, cần bao nhiêu tuần để phát hiện +5 điểm %? Chạy 4 tuần thì phát hiện
   được mức nào? Mức đó có thực tế với một email chào mừng không?
3. Thời lượng ở trên mới là thời gian **gom khách**. Metric 60 ngày thì sớm nhất khi nào có kết quả?
4. Nếu không thể chạy đủ, bạn đề xuất gì: đổi metric (cửa sổ ngắn hơn), mở rộng đối tượng (mọi khách đang
   hoạt động, không chỉ khách mới), chấp nhận MDE lớn hơn, hay không A/B mà đo bằng holdout dài hạn?

Viết ba câu trả lời cho trưởng nhóm CRM: được hay không, vì sao (con số), và bạn đề xuất gì thay thế.

Xong thì mở app, làm phần **trắc nghiệm** của bài.